In [ ]:
# 03 — Clusterização de Produtos

# Contextualização

Em bases de dados com grande diversidade de produtos, é comum observar comportamentos de vendas bastante distintos,
influenciados por fatores como sazonalidade, frequência de consumo e volume médio de demanda.

Nesse contexto, a clusterização surge como uma estratégia para identificar grupos homogêneos, permitindo análises mais refinadas
e possibilitando a construção de modelos preditivos mais especializados.


In [ ]:
# Objetivos

Este notebook tem como objetivo aplicar técnicas de aprendizado não supervisionado para agrupar produtos com padrões semelhantes de venda,
utilizando o algoritmo K-Means.

Busca-se identificar estruturas latentes nos dados, reduzindo a complexidade do problema e fornecendo subsídios para a modelagem preditiva segmentada.


In [ ]:
# Justificativa da Clusterização

A aplicação de modelos globais em bases altamente heterogêneas pode resultar em previsões imprecisas,
uma vez que um único modelo precisa aprender padrões muito distintos simultaneamente.

Ao agrupar produtos com comportamentos semelhantes, torna-se possível treinar modelos específicos para cada grupo,
aumentando a capacidade de generalização e reduzindo erros de previsão.


In [ ]:
# Importação e Setup

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


In [ ]:
# Extração de Atributos para Clusterização

Para caracterizar o comportamento de cada produto, foram extraídas estatísticas descritivas capazes de sintetizar padrões de consumo ao longo do tempo.

def extract_product_features(df):
    grouped = df.groupby("product_id")["Quantity"]

    features = pd.DataFrame({
        "mean": grouped.mean(),
        "std": grouped.std(),
        "median": grouped.median(),
        "max": grouped.max(),
        "min": grouped.min(),
        "cv": grouped.std() / grouped.mean(),
        "non_zero_ratio": grouped.apply(lambda x: (x > 0).mean())
    })

    return features.dropna()


In [ ]:
# Normalização

Como os atributos extraídos possuem escalas distintas, aplica-se a normalização para evitar vieses no processo de agrupamento.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(product_features)


In [ ]:
# Definição do Número Ideal de Clusters

inertias = []
K = range(2, 11)

for k in K:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.plot(K, inertias, marker='o')
plt.xlabel("Número de clusters")
plt.ylabel("Inércia")
plt.title("Método do Cotovelo")
plt.show()

sil_scores = []

for k in K:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

plt.plot(K, sil_scores, marker='o')
plt.xlabel("Número de clusters")
plt.ylabel("Silhouette Score")
plt.title("Análise de Silhouette")
plt.show()


In [ ]:
# Aplicação do K- Means

k = 4  # definido após análise
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

product_features["cluster"] = clusters


In [ ]:
# Análise dos Clusters

product_features.groupby("cluster").mean()

product_features["cluster"].value_counts()


A análise estatística dos grupos formados revela padrões distintos de consumo, como produtos de alta rotatividade, baixa demanda,
comportamento esporádico e alta variabilidade.

Essa segmentação possibilita a aplicação de estratégias específicas de modelagem para cada grupo.


In [ ]:
# Integração com o Pipeline de Modelagem

Os clusters obtidos são incorporados ao pipeline geral, permitindo a separação dos dados por grupo
e o treinamento de modelos preditivos especializados para cada segmento.

product_features[["cluster"]].to_csv("../data/processed/product_clusters.csv")

